# Train PPO on CarRacing-v3 (Kaggle GPU)

This notebook trains a PPO agent with a CNN policy on CarRacing-v3 using Stable-Baselines3.

**Setup:** Settings → Accelerator → GPU T4 x1, Internet → ON

**Key trick: frame stacking.** We feed the last 4 frames as a single input (4×96×96 = 12 channels)
so the CNN can see motion between frames — without this, the network has no way to tell
how fast the car is going or which direction it's turning, and PPO plateaus around 400.

**Training budget:** 2M steps, ~6 hours on T4 (well within Kaggle's 12-hour limit).

**Output:** `ppo_carracing.zip` (~25 MB) — download it from the Output panel when done,
then place it at `dl/ppo_carracing.zip` in the local repo.

In [ ]:
!pip install -q "gymnasium[box2d]==1.2.3" "stable-baselines3==2.8.0" swig

In [ ]:
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecTransposeImage, VecFrameStack

TOTAL_STEPS = 2_000_000   # ~6 hours on T4, expected return 700-900
N_ENVS = 8
N_STACK = 4               # feed last 4 frames so the CNN can see motion

vec_env = make_vec_env(
    "CarRacing-v3",
    n_envs=N_ENVS,
    env_kwargs={"continuous": True},
)
vec_env = VecTransposeImage(vec_env)
vec_env = VecFrameStack(vec_env, n_stack=N_STACK)

model = PPO(
    "CnnPolicy",
    vec_env,
    verbose=1,
    learning_rate=1e-4,
    n_steps=512,
    batch_size=128,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    vf_coef=0.5,
    max_grad_norm=0.5,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print(f"Training for {TOTAL_STEPS:,} steps on {N_ENVS} parallel envs, frame stack = {N_STACK}")
model.learn(total_timesteps=TOTAL_STEPS, progress_bar=False)

model.save("/kaggle/working/ppo_carracing")
print("Saved to /kaggle/working/ppo_carracing.zip")

## Sanity check: run seed 0 with the trained agent

In [ ]:
import gymnasium as gym
from stable_baselines3.common.vec_env import DummyVecEnv, VecTransposeImage, VecFrameStack

# Eval needs the same frame-stack wrapping the model was trained with.
def make_eval_env(seed):
    def _make():
        env = gym.make("CarRacing-v3", continuous=True)
        env.reset(seed=seed)
        return env
    vec = DummyVecEnv([_make])
    vec = VecTransposeImage(vec)
    vec = VecFrameStack(vec, n_stack=4)
    return vec

eval_env = make_eval_env(seed=0)
obs = eval_env.reset()
total = 0.0
done = False
while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, dones, infos = eval_env.step(action)
    total += reward[0]
    done = dones[0]
eval_env.close()
print(f"seed=0 return={total:.1f}")

## Quick eval on seeds 0-9

In [ ]:
import numpy as np

model_eval = PPO.load("/kaggle/working/ppo_carracing", device="cuda" if torch.cuda.is_available() else "cpu")

returns = []
for seed in range(10):
    eval_env = make_eval_env(seed=seed)
    obs = eval_env.reset()
    total = 0.0
    done = False
    while not done:
        action, _ = model_eval.predict(obs, deterministic=True)
        obs, reward, dones, infos = eval_env.step(action)
        total += reward[0]
        done = dones[0]
    eval_env.close()
    returns.append(total)
    print(f"seed={seed}  return={total:.1f}")

print(f"\nmean={np.mean(returns):.1f}  std={np.std(returns):.1f}  "
      f"good(>=700)={sum(r >= 700 for r in returns)}  "
      f"full_lap(>=900)={sum(r >= 900 for r in returns)}")

## Download

The trained checkpoint is at `/kaggle/working/ppo_carracing.zip` in the **Output** panel on the right.

Download it and place it at `dl/ppo_carracing.zip` in the local repo.